In [29]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report 
df_matches = pd.read_csv('../data/processed/matches_clean.csv')


In [20]:
df_matches['result'] = np.where(
    df_matches['home_team_goals'] > df_matches['away_team_goals'] ,'Home Win',
    np.where(
        df_matches['home_team_goals'] < df_matches['away_team_goals'],'Away Wins',
        'Draw'
    )
)
df_matches['result'].value_counts()

result
Home Win     486
Draw         190
Away Wins    174
Name: count, dtype: int64

In [21]:
# Calculate each team's stats
all_teams = pd.concat(
   [
       df_matches['home_team_name'],
       df_matches['away_team_name']
   ] 
).unique()
print(f'Total unique team: {len(all_teams)}')

Total unique team: 83


In [22]:
team_stats = []
for team in all_teams:
    home = df_matches[df_matches['home_team_name'] == team]
    away = df_matches[df_matches["away_team_name"] == team]
    total = len(home) + len(away)

    if total == 0:
        continue
    home_wins = len(home[home['result']== 'Home Win'])
    away_wins = len(away[away['result']== 'Away Win'])
    total_wins = home_wins + away_wins

    goals_scored = home['home_team_goals'].sum() + away['away_team_goals'].sum()
    goals_conceded = home['away_team_goals'].sum() + away['home_team_goals'].sum()

    team_stats.append({
        'team': team,
        'win_rate': total_wins / total,
        'avg_goals_scored': goals_scored /total,
        'avg_goals_conceded': goals_conceded / total
    })

df_team_stats = pd.DataFrame(team_stats)
print(df_team_stats.head(10))
df_team_stats = df_team_stats.set_index('team')
print(df_team_stats.loc['Brazil'])

         team  win_rate  avg_goals_scored  avg_goals_conceded
0      France  0.311475          1.770492            1.180328
1         USA  0.176471          1.117647            1.882353
2  Yugoslavia  0.351351          1.621622            1.243243
3     Romania  0.238095          1.428571            1.523810
4   Argentina  0.506173          1.641975            1.049383
5       Chile  0.264706          1.205882            1.470588
6     Uruguay  0.307692          1.538462            1.403846
7      Brazil  0.546296          2.083333            1.055556
8    Paraguay  0.148148          1.111111            1.407407
9     Austria  0.344828          1.482759            1.620690
win_rate              0.546296
avg_goals_scored      2.083333
avg_goals_conceded    1.055556
Name: Brazil, dtype: float64


In [23]:
features= []

for _,row in df_matches.iterrows():
    home = row['home_team_name']
    away = row['away_team_name']

    if home not in df_team_stats.index or away not in df_team_stats.index:
        continue

    hs = df_team_stats.loc[home]
    as_ = df_team_stats.loc[away]

    features.append({
        'home_win_rate': hs['win_rate'],
        'home_avg_scored': hs['avg_goals_scored'],
        'home_avg_conceded': hs['avg_goals_conceded'],
        'away_win_rate': as_['win_rate'],
        'away_avg_scored': as_['avg_goals_scored'],
        'away_avg_conceded': as_['avg_goals_conceded'],
        'result': row['result']
    })

df_feature = pd.DataFrame(features)
print(df_feature.shape)
print(df_feature.head())

(850, 7)
   home_win_rate  home_avg_scored  home_avg_conceded  away_win_rate  \
0       0.311475         1.770492           1.180328       0.148148   
1       0.176471         1.117647           1.882353       0.255814   
2       0.351351         1.621622           1.243243       0.546296   
3       0.238095         1.428571           1.523810       0.266667   
4       0.506173         1.641975           1.049383       0.311475   

   away_avg_scored  away_avg_conceded    result  
0         1.074074           1.740741  Home Win  
1         1.255814           1.581395  Home Win  
2         2.083333           1.055556  Home Win  
3         1.266667           2.066667  Home Win  
4         1.770492           1.180328  Home Win  


In [24]:
X = df_feature.drop(columns=['result'])
y = df_feature['result']

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.2%}')
print('\nDetailed Report:')
print(classification_report(y_test, y_pred))

Accuracy: 58.82%

Detailed Report:
              precision    recall  f1-score   support

   Away Wins       0.61      0.45      0.52        38
        Draw       0.26      0.14      0.19        35
    Home Win       0.63      0.80      0.71        97

    accuracy                           0.59       170
   macro avg       0.50      0.46      0.47       170
weighted avg       0.55      0.59      0.56       170



Prediciting brazil vs france where home team is brazil and away is france

In [27]:
brazil = df_team_stats.loc['Brazil']
france = df_team_stats.loc['France']

match = pd.DataFrame([{
    'home_win_rate': brazil['win_rate'],
    'home_avg_scored': brazil['avg_goals_scored'],
    'home_avg_conceded': brazil['avg_goals_conceded'],
    'away_win_rate': france['win_rate'],
    'away_avg_scored': france['avg_goals_scored'],
    'away_avg_conceded': france['avg_goals_conceded'],
}])

prediction = model.predict(match)
probability = model.predict_proba(match)

print(f'Predicted Result :{prediction[0]}')
print(f'Probabilites: {dict(zip(model.classes_, probability[0].round(2)))}')

Predicted Result :Away Wins
Probabilites: {'Away Wins': np.float64(0.44), 'Draw': np.float64(0.24), 'Home Win': np.float64(0.32)}


In [30]:
with open('../models/worldcup_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('Model saved! ✅')

Model saved! ✅
